# Gloss-Free Sign Language Translation (ASL-to-English)
## Portfolio Training Notebook for Kaggle GPUs

This notebook provides a complete pipeline to clone the landmark-based ASL translation repository, set up the dependency environment, validate keypoint datasets, run sanitization unit tests, preprocess the dataset, train our hybrid **Conformer-T5 model**, and export the trained model to ONNX for fast inference.

## 1. Setup, Environment Validation, and Sanity Checks

In [ ]:
# Clone or update the repository
import os
if not os.path.exists('/kaggle/working/gloss-free-asl-translation'):
    !git clone https://github.com/yyouretoast/gloss-free-asl-translation.git
    %cd /kaggle/working/gloss-free-asl-translation
else:
    %cd /kaggle/working/gloss-free-asl-translation
    !git checkout -- requirements.txt
    !git pull


In [ ]:
# Filter out torch/torchvision to keep Kaggle's GPU-optimized pre-installs
!sed -i '/torch/d' requirements.txt
!pip install -r requirements.txt
!pip install -e .


In [ ]:
# Run environment validation to verify GPU availability and dependencies
!python -m scripts.check_environment


### 1.1 Codebase Sanity Verification
Verify the codebase sanity by running our test suite. This ensures that the Conformer-T5 model architecture, datasets, and preprocessing pipelines are fully operational before initiating long training runs.

In [ ]:
# Run unit tests (excluding slow dataset integration tests)
!pytest tests/ -v -m "not slow"


## 2. Environment Configuration (Optional HF Token)

Set your Hugging Face token to enable faster downloads of the `t5-small` model and avoid rate limits.

In [ ]:
import os
# Set your HF Token if available
# os.environ["HF_TOKEN"] = "your_huggingface_token_here"

## 3. Real-World Dataset Profiling & Validation

Before training, run the profile validation script to inspect the tracking dropouts and sequence lengths of the coordinate datasets (e.g., How2Sign or YouTube-ASL).

In [ ]:
# Run dataset validation profile (point --data_dir to the actual coordinates directory)
# For YouTube-ASL:
# !python -m src.validate_dataset --data_dir /kaggle/input/youtube-asl/landmarks

# For How2Sign (auto-resolving paths):
import os
data_dir = next((p for p in [
    '/kaggle/input/datasets/nazarboholii/how2sign/train_2D_keypoints/openpose_output/json',
    '/kaggle/input/how2sign/train_2D_keypoints/openpose_output/json',
    '/kaggle/input/how2sign-keypoints/train_2D_keypoints/openpose_output/json',
    '/kaggle/input/datasets/nazarboholii/how2sign-keypoints/train_2D_keypoints/openpose_output/json'
] if os.path.exists(p)), '/kaggle/input/datasets/nazarboholii/how2sign/train_2D_keypoints/openpose_output/json')
!python -m src.validate_dataset --data_dir {data_dir}


In [ ]:
# Dataset structure audit utility (adds flexibility for How2Sign or custom data)
import os
import glob
import numpy as np

# Choose your input dataset path (auto-resolving paths)
for path in [
    '/kaggle/input/datasets/nazarboholii/how2sign',
    '/kaggle/input/how2sign',
    '/kaggle/input/how2sign-keypoints',
    '/kaggle/input/datasets/nazarboholii/how2sign-keypoints'
]:
    if os.path.exists(path):
        input_dir = path
        break
else:
    input_dir = '/kaggle/input/datasets/nazarboholii/how2sign'

# Fast inspection without slow recursive globbing
npz_files = []
npy_files = []
json_dirs = []

# Check candidate OpenPose directory directly
json_cand = os.path.join(input_dir, "train_2D_keypoints/openpose_output/json")
if os.path.exists(json_cand):
    # list directories under this path
    json_dirs = [os.path.join(json_cand, d) for d in os.listdir(json_cand) if os.path.isdir(os.path.join(json_cand, d))]

# If no json dirs, check if there are NPZs in input_dir directly
if not json_dirs:
    if os.path.exists(input_dir):
        # Check direct files
        npz_files = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith('.npz')]
        npy_files = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith('.npy')]

print(f"Total .npz files found: {len(npz_files)}")
print(f"Total .npy files found: {len(npy_files)}")
print(f"Total OpenPose directories found: {len(json_dirs)}")

test_file = None
if npz_files:
    test_file = npz_files[0]
    print(f"\nAuditing NPZ file: {test_file}")
    with np.load(test_file) as data:
        print("Keys in file:", list(data.files))
        for key in data.files:
            print(f" - Key: '{key}', Shape: {data[key].shape}")
elif npy_files:
    test_file = npy_files[0]
    print(f"\nAuditing NPY file: {test_file}")
    data = np.load(test_file)
    print(f"Shape: {data.shape}")
elif json_dirs:
    test_dir = json_dirs[0]
    print(f"\nAuditing OpenPose Directory: {test_dir}")
    from src.utils.io_utils import load_openpose_directory
    landmarks = load_openpose_directory(test_dir)
    for k, v in landmarks.items():
        print(f" - Key: '{k}', Shape: {v.shape}")
else:
    print("\nNo coordinate files found. Listing directory structure:")
    for root, dirs, files in os.walk(input_dir):
        print(f"Directory: {root}")
        for file in files[:5]:
            print(f"  - {file}")


## 4. High-Throughput Preprocessing (Raw JSON to Compressed NPZ)

To avoid severe CPU-disk latency bottlenecks during training, convert the raw OpenPose JSON frame files into compressed `.npz` files in parallel using all available CPU cores. This reduces step latency from ~12.5 seconds to milliseconds, ensuring the GPU is never starved.

In [ ]:
# Run parallel dataset preprocessing
!python scripts/preprocess_how2sign.py --input-dir {input_dir} --output-dir /tmp/how2sign_npz


### 4.2 Persist Preprocessed Dataset for Future Runs (Optional)

Since preprocessing the 31,000+ folders of How2Sign takes approximately 2 hours, zipping the preprocessed `.npz` files and saving them to Kaggle's `/kaggle/working` directory allows you to download them or persist them as a Kaggle output dataset. This lets you skip the preprocessing phase entirely in future runs.

In [ ]:
# Zip the preprocessed SSD folder to Kaggle's persistent working directory
!zip -q -r /kaggle/working/how2sign_npz.zip /tmp/how2sign_npz
print("Zipped preprocessed files to /kaggle/working/how2sign_npz.zip")

## 5. End-to-End Conformer-T5 Model Training

This runs our streamlined end-to-end `Conformer -> T5-Small` translation pipeline using the preprocessed NPZ dataset. Adjust batch size and learning rate as needed.

In [ ]:
# --- Option A: Load TensorBoard visualization ---
%load_ext tensorboard
%tensorboard --logdir results/checkpoints/runs


In [ ]:
# --- Option A: How2Sign Training (NPZ-based) ---
import os
data_dir = "/tmp/how2sign_npz"

metadata_file = next((p for p in [
    '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv',
    '/kaggle/input/how2sign/how2sign_realigned_train.csv',
    '/kaggle/input/how2sign-keypoints/how2sign_realigned_train.csv',
    '/kaggle/input/datasets/nazarboholii/how2sign-keypoints/how2sign_realigned_train.csv'
] if os.path.exists(p)), '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv')

# Phase 1: Train model with face landmarks enabled (411 dimensions)
!python -m src.train --epochs 50 --batch_size 8 --lr 1e-4 --data_dir {data_dir} --metadata_file {metadata_file} --resume_from_checkpoint latest

# Phase 2: Train model with face landmarks disabled (ablation study - 201 dimensions)
# !python -m src.train --epochs 50 --batch_size 8 --lr 1e-4 --data_dir {data_dir} --metadata_file {metadata_file} --no_face --resume_from_checkpoint latest

# --- Option B: YouTube-ASL Training ---
# Phase 1: Train model with face landmarks enabled (534 dimensions)
# !python -m src.train --epochs 50 --batch_size 8 --lr 1e-4 --data_dir /kaggle/input/youtube-asl/landmarks --metadata_file /kaggle/input/youtube-asl/metadata.csv --resume_from_checkpoint latest

# Phase 2: Train model with face landmarks disabled (ablation study - 258 dimensions)
# !python -m src.train --epochs 50 --batch_size 8 --lr 1e-4 --data_dir /kaggle/input/youtube-asl/landmarks --metadata_file /kaggle/input/youtube-asl/metadata.csv --no_face

## 6. Model Optimization & Export (Conformer Encoder to ONNX)

After the 50-epoch training finishes, trace and export the Conformer Encoder component directly to ONNX. This saves the exported `/kaggle/working/conformer_encoder.onnx` file directly into your output files for fast, resource-efficient web demo or edge deployment, bypassing T5's heavy autogeneration loops where needed.

In [ ]:
# Trace and export the best checkpoint to ONNX
import os

checkpoint_dirs = sorted(glob.glob("results/checkpoints/checkpoint-*"), key=lambda x: int(x.split("-")[-1]))
if checkpoint_dirs:
    best_checkpoint = checkpoint_dirs[-1]
    model_bin = os.path.join(best_checkpoint, "model.safetensors")
    if not os.path.exists(model_bin):
        model_bin = os.path.join(best_checkpoint, "pytorch_model.bin")
    
    # Resolve input dimension dynamically: 411 for face-enabled How2Sign, 201 for face-disabled ablation run
    # For YouTube-ASL, face-enabled is 534, face-disabled is 258.
    input_dim = 411  # CHANGE this value if you ran training with --no_face (e.g. 201) or on YouTube-ASL!
    
    print(f"Exporting encoder from {model_bin} to ONNX with input_dim={input_dim}...")
    !python -X utf8 scripts/export_onnx.py --input-dim {input_dim} --model-path {model_bin} --output /kaggle/working/conformer_encoder.onnx
else:
    print("No checkpoints found to export!")

## 7. Checkpoint Archival & Retrieval

Zip the complete checkpoints folder so you can download the raw PyTorch weights (`pytorch_model.bin`, `config.json`) and back them up locally or register them as a Kaggle model version.

In [ ]:
# Zip all checkpoints for easy download
!zip -q -r /kaggle/working/checkpoints.zip results/checkpoints
print("Zipped checkpoints to /kaggle/working/checkpoints.zip")

## 8. Quantitative Evaluation & Qualitative Inference

After training, run quantitative evaluation to compute BLEU-4 and WER metrics, and perform qualitative inference to verify the translation outputs on selected validation examples.

In [ ]:
# --- 8a. Run Quantitative Evaluation ---
import glob
import os

checkpoint_dirs = sorted(glob.glob("results/checkpoints/checkpoint-*"), key=lambda x: int(x.split("-")[-1]))
if checkpoint_dirs:
    best_checkpoint = checkpoint_dirs[-1]
    
    # Resolve metadata & dataset paths dynamically
    data_dir = "/tmp/how2sign_npz"
    metadata_file = next((p for p in [
        '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv',
        '/kaggle/input/how2sign/how2sign_realigned_train.csv',
        '/kaggle/input/how2sign-keypoints/how2sign_realigned_train.csv',
        '/kaggle/input/datasets/nazarboholii/how2sign-keypoints/how2sign_realigned_train.csv'
    ] if os.path.exists(p)), '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv')
    
    # Install evaluation metrics libraries if not present
    !pip install -q jiwer sacrebleu
    
    print(f"Evaluating best checkpoint: {best_checkpoint} against {metadata_file}...")
    !python -m scripts.evaluate --checkpoint {best_checkpoint} --data-dir {data_dir} --metadata {metadata_file}
else:
    print("No checkpoints found to evaluate!")


### 8.2 Qualitative Sample Inference
Load the trained model and execute translation on sample validation landmarks directly in the notebook to inspect translation quality side-by-side with ground-truth sentences.

In [ ]:
# --- 8b. Run Qualitative Sample Inference ---
import os
import glob
import torch
import numpy as np
from transformers import T5TokenizerFast
from src.models.translation_model import ASLTranslationModel
from src.dataset import ASLLandmarkDataset

device = "cuda" if torch.cuda.is_available() else "cpu"

# Locate best checkpoint
checkpoint_dirs = sorted(glob.glob("results/checkpoints/checkpoint-*"), key=lambda x: int(x.split("-")[-1]))
if checkpoint_dirs:
    best_checkpoint = checkpoint_dirs[-1]
    model_path = os.path.join(best_checkpoint, "model.safetensors")
    if not os.path.exists(model_path):
        model_path = os.path.join(best_checkpoint, "pytorch_model.bin")
    
    # Paths
    data_dir = "/tmp/how2sign_npz"
    metadata_file = next((p for p in [
        '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv',
        '/kaggle/input/how2sign/how2sign_realigned_train.csv',
        '/kaggle/input/how2sign-keypoints/how2sign_realigned_train.csv',
        '/kaggle/input/datasets/nazarboholii/how2sign-keypoints/how2sign_realigned_train.csv'
    ] if os.path.exists(p)), '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv')
    
    print(f"Loading best checkpoint from {model_path}...")
    
    # Load metadata
    from src.utils.metadata import load_metadata
    metadata, _ = load_metadata(metadata_file)
    
    # Load dataset & tokenizer
    tokenizer = T5TokenizerFast.from_pretrained("t5-small")
    dataset = ASLLandmarkDataset(
        data_dir=data_dir,
        metadata_dict=metadata,
        max_len=150,
        include_face=True,
        normalize=True,
        skip_empty_labels=True
    )
    
    if len(dataset) > 0:
        # Load model
        input_dim = dataset[0]['features'].shape[1]
        model = ASLTranslationModel(
            input_dim=input_dim,
            d_model=512,
            t5_model_name="t5-small",
            num_layers=4,
            num_heads=4,
            kernel_size=31
        )
        if model_path.endswith(".safetensors"):
            try:
                from safetensors.torch import load_file
                state_dict = load_file(model_path, device="cpu")
            except ImportError:
                raise ImportError("Found model.safetensors, but safetensors package is not installed. Please run `pip install safetensors`.")
        else:
            state_dict = torch.load(model_path, map_location="cpu", weights_only=True)
        model.load_state_dict(state_dict)
        model = model.to(device)
        model.eval()
        
        import pandas as pd
        # Test on a few samples
        print("\n--- Qualitative Predictions ---")
        num_samples = min(5, len(dataset))
        targets = []
        predictions = []
        for i in range(num_samples):
            sample = dataset[i]
            features = sample['features'].unsqueeze(0).to(device)  # Add batch dim
            attention_mask = torch.ones((1, features.shape[1]), dtype=torch.float32, device=device)
            
            with torch.no_grad():
                output_ids = model.generate(
                    input_features=features,
                    attention_mask=attention_mask,
                    max_new_tokens=30
                )
            pred_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            targets.append(sample['text'])
            predictions.append(pred_text)
        
        # Display in a beautiful Pandas DataFrame table
        results_df = pd.DataFrame({
            "Sample": [f"#{i+1}" for i in range(num_samples)],
            "Target (Ground Truth)": targets,
            "Predicted Translation": predictions
        })
        from IPython.display import display, HTML
        display(HTML(results_df.to_html(index=False)))
else:
    print("No checkpoints found for qualitative inference!")
